# Import Required Libraries

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# to load env variables
from dotenv import load_dotenv
from openai import OpenAI # OpenAI llm
from agents import Agent, Runner, trace, handoff, OpenAIChatCompletionsModel, function_tool # OpenAI SDK primitives
from openai.types.responses import ResponseTextDeltaEvent

from openai import AsyncOpenAI

# asynio
import asyncio

# Decorator Packages
from IPython.display import display, Markdown
from typing import Dict, List


# SendGrid
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Email, To, Content


Exception in thread Thread-4 (_run):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/Volumes/Transcend/DeepLearning/.venv_agentic_ai/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Volumes/Transcend/DeepLearning/.venv_agentic_ai/lib/python3.13/site-packages/agents/tracing/processors.py", line 234, in _run
    self._export_batches(force=False)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/Volumes/Transcend/DeepLearning/.venv_agentic_ai/lib/python3.13/site-packages/agents/tracing/processors.py", line 267, in _export_batches
    self._exporter.expor

In [2]:
load_dotenv(override = True)

True

In [3]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
SENDGRID_API_KEY = os.getenv("TWILIO_API_KEY")
FROM_EMAIL = os.getenv("FROM_EMAIL")
TO_EMAIL = os.getenv("TO_EMAIL")

# Let's create three agents 

- We need to define instructions what the agent do - you can think of system prompt
- Call openai sdk agent by defining name, instructions and model

In [4]:
instructions1  = "You are a sales agent working for Unilever, a company that produces wide range of products related to health care, deodrants & fagrance, icecream, laundry etc., . \
    You write professional, serious cold emails"

instructions2 = "You are a humorous sales agent working for Unilever, a company that produces wide range of products related to health care, deodrants & fagrance, icecream, laundry etc., . \
    You write witty, engaging cold emails that are likely to get a response"

instructions3 = "You are a humorous sales agent working for Unilever, a company that produces wide range of products related to health care, deodrants & fagrance, icecream, laundry etc., . \
    You write concise, to the point cold emails"


In [5]:
# Professional Sales Agent
sales_agent1 = Agent(name = "Professional Sales Agent", 
      instructions = instructions1,
      model = "gpt-4o-mini")


# Engaging Sales Agent
sales_agent2 = Agent(name = "Engaging Sales Agent", 
                     instructions = instructions2,
                     model = "gpt-4o-mini")

# Busy Sales Agent
sales_agent3 = Agent(name = "Busy Sales Agent", 
                     instructions = instructions3,
                     model = "gpt-4o-mini")

In [ ]:
result = Runner.run_streamed(sales_agent1, input = "Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = "", flush = True)

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"): # Trace helps you record the progress of the calls in 
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    
# 

In [ ]:
outputs = [result.final_output for result in results] # results holds list of response from different llms

for output in outputs:
    print(output + "\n\n")

## Sales Picker Agent

- The job of sales picker agent is to identify best cold emails and select that
- Now, we are going to use Gemini model as the evaluator

You can track the response : https://platform.openai.com/traces

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/" # url endpoint

gemini_client = AsyncOpenAI(base_url = GEMINI_BASE_URL, api_key = GOOGLE_API_KEY)
gemini_model = OpenAIChatCompletionsModel(model = "gemini-2.0-flash", openai_client = gemini_client)

In [7]:
instructions4 = "You pick the best cold sales email from the given options. \
    Image you are customer and pick the oone you are most likely to respond to. \
        Do not give an explanation; reply with the email"

sales_picker_agent = Agent(
    name = "Sales Picker Agent", 
    instructions = instructions4,
    model = gemini_model
)

In [8]:
input_message = "Write a cold email message"
with trace("Sales_Picker_Response"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, input_message),
        Runner.run(sales_agent2, input_message),
        Runner.run(sales_agent3, input_message)
    )

    outputs = [f"Generated by Sales Agent {index} \n\n"+ result.final_output
                 for index, result in zip(list(range(0, len(results))), results)]

    # Append the output of each response of individual agent 
    emails = "Cold email messages : \n\n".join(outputs)    

    best_email = await Runner.run(sales_picker_agent, emails)

    print(f"Best sales email: \n {best_email.final_output}")

Best sales email: 
 Generated by Sales Agent 2 



Observation:

- What we did is we generated the cold emails using three different agents, and we used evaluator agent to pick the response that has high chance of customer responding to it 

- Built in tracing helps us to visualize, debug and monitor your workflows, as well as use the OpenAI suite of evaluation, fine-tuning and distillation tools

# Let's see how we can utilize Tools


-  We will exploring Twilio SenderGrid as tool in this excerise
-  OpenAI SDK provides function tools wrapper that turns any python function into a tool, with automatic schema generation and utilizes Pydantic-powered validation
-  The main objective tool in Agents environment, the Agent uses the tool when it has perform certain operation and returns back the control to the Agent once it's operation is performed

In [ ]:
# The Operation we are talking about here is sending out best email that is generated by the Sales Picker Agent
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects"""
    sg = SendGridAPIClient(api_key=SENDGRID_API_KEY) 
    from_email = Email(FROM_EMAIL)
    to_email = To(TO_EMAIL)
    content = Content('text/plain', body)
    mail = Mail(from_email, to_email, "Sales Email", content) 
    # Get a JSON-ready representation of the mail object
    mail_json = mail.get()

    # Send a HTTP post request to /mail/send
    response = sg.client.mail.send.post(request_body = mail_json)
    return {'status':'success'}

In [ ]:
send_email 
# As we said this Function_Tool wrapper converted the function into tooll and create json object automatically which actually lifts lots of load on the developer
# It took doc string object as the description of the tool

#### Agent as Tool
- We can also convert the agent as tool
- We will be converting all the three agents as tool

In [9]:
description = "Write a cold sales email"
description_1 = "Pick the best cold sales email"
tool1 = sales_agent1.as_tool(tool_name = 'sales_agent1', tool_description= description)
tool2 = sales_agent2.as_tool(tool_name = "sales_agent2", tool_description= description)
tool3 = sales_agent3.as_tool(tool_name = "sales_agent3", tool_description= description)
tool4 = sales_picker_agent.as_tool(tool_name = "sales_picker_agent",tool_description=description_1)

tools = [tool1, tool2, tool3, tool4]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10ff6f380>, strict_json_schema=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d525800>, strict_json_schema=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent

# And now it's time for our Sales Manager - our planning Agent

In [10]:
instructions = "You are a sales manager working for ComplAI. You use the tools given to you to operate cold sales emails. \
    You never generate sales emails yourself; you always use the tools.\
        You try all 3 sales_agent tools once before choosing the best one. \
            You pick the single best email using the tool"

sales_manager = Agent(name="Sales Manager", 
                      instructions=instructions, 
                      tools = tools, 
                      model= gemini_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales Manager Sending Out Email"):
    result = await Runner.run(sales_manager, message)

In [49]:
display(Markdown(result.final_output))

Okay, I have evaluated all three options and chosen the best one. Here is the cold sales email I recommend:

Subject: Ready to Partner Up and Unleash Some Unilever Magic? ✨

Dear CEO,

I hope this email finds you knee-deep in success and sipping the world’s finest coffee (perhaps made with a little help from our coffee creamers? 😉).

I’m writing to you from Unilever, the land where health care, deodorants, ice cream, and laundry detergents all gather for a grand meeting to discuss world domination—one product at a time!

Imagine this: Your brand, fortified with our top-tier products, together taking over grocery shelves and leaving the competition wondering, “Who are those superstars?”

We can help your brand smell better (with a little fragrant persuasion), stay fresher (because who doesn’t love that?), and even taste a bit sweeter (ice cream, anyone?). Plus, if your laundry needs a little love, we can lighten the load (literally!).

Let’s schedule a chat to discuss how we can blend our strengths like the perfect scoop of mint chocolate chip—refreshing, delightful, and thoroughly unforgettable.

Looking forward to your reply. If nothing else, I promise to bring the ice cream!

Best wishes and witty puns,

[Your Name]
[Your Position]
Unilever


- We saw how can we use Agents-as-tools and function_tools to wrap a python function which can also be used as tool

# Handoffs

- Handoffs and Agents-as-tools are similar:

    - In both cases, an Agent can collaborate with other Agent

    - With tools, controls passes back
    
    - With handoffs, control passes across  

In [80]:
# Subject Email Writer Tool
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response"

subject_agent = Agent(name = "subject email agent", instructions = subject_instructions, model = "gpt-4o-mini")
tool5 = subject_agent.as_tool(tool_name = "subject_writer", tool_description= "Write a subject for a cold email")

# HTML Email Converter Tool
html_instructions = "You can convert a text email body to an HTMl email body. \
You are given a text email body which might have some markdown and you need to \
convert it to a HTML email body with simple, clear, compelling layout and design."

html_agent = Agent(name = "html email body converter", instructions = html_instructions, model = "gpt-4o-mini")
tool6 = html_agent.as_tool(tool_name = "html_converter", tool_description = "Convert text mail body to html mail body")

In [ ]:
tools_email = [tool5, tool6] 

In [ ]:
instruction_emailer = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the tool to convert the text mail body to HTML mail body. \
And send the response"

emailer_agent = Agent(
    name = "Emailer Agent",
    instructions = instruction_emailer, 
    model = gemini_model,
    tools = tools_email,
    handoff_description= "Convert an email to HTML and send it"
)

#### Now, we have three tools and 1 handoff


In [ ]:
sales_manager_instructions = "You are a sales manager working for ComplAI. You use the tools given to you to operate cold sales emails. \
You never generate sales emails yourself; you always use the tools.\
You try all 3 sales_agent tools once before choosing the best one. \
You pick the single best email using the tools. After picking the email, you handoff to Emailer Agent to format and send the response "

sales_manager_agent = Agent(
    name = "Sales Manager Agent",
    instructions = sales_manager_instructions,
    tools = tools, 
    handoffs = [handoff(agent = emailer_agent)],
    model =gemini_model
)

In [100]:
with trace("Automated SDR New"):
    result = await Runner.run(sales_manager_agent, input= "Send out a cold sales email addressed to Dear CEO from Sans")

In [101]:
result.final_output

''